In [23]:
import pandas as pd
df = pd.read_csv("raw_skills_100.csv")
df

,Role,Skills
0,DevOps Engineer,Terraform Docker Linux CI/CD Kubernetes
1,Data Scientist,MachineLearning TensorFlow Statistics Python SQL
2,Cybersecurity Analyst,Linux Networking CyberSecurity Python Wireshark
3,Frontend Developer,CSS JavaScript HTML Angular React
4,Data Scientist,SQL TensorFlow MachineLearning Statistics NumPy
...,...,...
95,Web Developer,HTML Bootstrap NodeJS CSS React
96,Frontend Developer,VueJS React HTML Angular CSS
97,Web Developer,React Bootstrap CSS NodeJS JavaScript
98,DevOps Engineer,Linux Docker Terraform CI/CD Kubernetes


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

def preprocess_text(text):
    # Convert to lowercase and remove spaces/special characters for better matching
    if not isinstance(text, str): return ""
    return re.sub(r'[^a-z0-9]', '', text.lower())

# -----------------------------
# User Input
# -----------------------------
user_input = input("Enter your skills: ")

# -----------------------------
# Process Skills
# -----------------------------
# We apply preprocessing to both the dataset and the user input
df['processed_skills'] = df['Skills'].apply(preprocess_text)
processed_user_input = preprocess_text(user_input)

all_skills = df["processed_skills"].tolist() + [processed_user_input]

# -----------------------------
# Convert Text to TF-IDF Matrix
# -----------------------------
# Using analyzer='char' and ngram_range lets us match 'MachineLearning' with 'Machine Learning'
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5))
tfidf_matrix = vectorizer.fit_transform(all_skills)

# -----------------------------
# Compute Similarity
# -----------------------------
similarity = cosine_similarity(
    tfidf_matrix[-1],
    tfidf_matrix[:-1]
)

# -----------------------------
# Add Similarity Scores
# -----------------------------
df["Similarity"] = similarity.flatten()

# -----------------------------
# Sort Recommendations
# -----------------------------
recommendations = df.sort_values(
    by="Similarity",
    ascending=False
)

# -----------------------------
# Display Results
# -----------------------------
print(f"\nTop Recommended Roles for '{user_input}':\n")

for index, row in recommendations.head(5).iterrows():
    print(f"Role: {row['Role']}")
    print(f"Skills Needed: {row['Skills']}")
    print(f"Match Score: {round(row['Similarity'] * 100, 2)}%")
    print("-" * 40)

Enter your skills: Python

Top Recommended Roles for 'Python':

Role: Backend Developer
Skills Needed: API Python Git Flask SQL
Match Score: 18.58%
----------------------------------------
Role: Backend Developer
Skills Needed: SQL API Flask Python Git
Match Score: 18.45%
----------------------------------------
Role: Backend Developer
Skills Needed: Python Git Flask API SQL
Match Score: 18.43%
----------------------------------------
Role: AI Engineer
Skills Needed: DeepLearning TensorFlow Python PyTorch NLP
Match Score: 17.38%
----------------------------------------
Role: Backend Developer
Skills Needed: Django API Flask Python Git
Match Score: 17.26%
----------------------------------------


In [25]:
pip install streamlit

In [26]:
%%writefile app.py
import streamlit as st
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Page Config
st.set_page_config(page_title="Tech stack Recommender", layout="wide")

# Preprocessing function
def preprocess_text(text):
    if not isinstance(text, str): return ""
    return re.sub(r'[^a-z0-9]', '', text.lower())

# Load Data
@st.cache_data
def load_data():
    df = pd.read_csv("raw_skills_100.csv")
    df['processed_skills'] = df['Skills'].apply(preprocess_text)
    return df

st.title("Tech Recommendation System")
st.write("Enter your skills below to find the best matching job roles.")

df = load_data()

# User Input
user_input = st.text_input("Your Skills (e.g., Python, Machine Learning, SQL)", "")

if user_input:
    processed_user_input = preprocess_text(user_input)
    all_skills = df["processed_skills"].tolist() + [processed_user_input]

    # Vectorizer logic
    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5))
    tfidf_matrix = vectorizer.fit_transform(all_skills)

    # Similarity
    similarity = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])
    df["Match Score"] = (similarity.flatten() * 100).round(2)

    # Display Results
    results = df.sort_values(by="Match Score", ascending=False).head(10)

    st.subheader(f"Top Recommendations for: {user_input}")
    st.dataframe(results[['Role', 'Skills', 'Match Score']], hide_index=True, use_container_width=True)
else:
    st.info("Please enter some skills to see recommendations.")

Overwriting app.py


### How to run the Streamlit app in Colab

Since Colab is a hosted environment, we need a way to expose the local Streamlit port. You can use **Localtunnel** to do this. Run the cell below to start your app and get a URL.

In [ ]:
!npm install -g localtunnel
import subprocess
import threading
import time

# Run streamlit in the background
def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py", "--server.port", "8501"])

thread = threading.Thread(target=run_streamlit)
thread.start()

# Wait for streamlit to start then open the tunnel
time.sleep(5)
print("Your Tunnel URL (Click the link below and enter the IP displayed below):")
!curl ipv4.icanhazip.com
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 4s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼Your Tunnel URL (Click the link below and enter the IP displayed below):
34.21.127.251
⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://slimy-socks-turn.loca.lt
